In [2]:
import pandas as pd
import numpy as np
import urllib.request
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasClassifier

Task 1: Data Acquisition and Loading

In [3]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigree', 'Age', 'Outcome']

df = pd.read_csv(url, header=None, names=columns)
print(f"Dataset loaded successfully! Shape: {df.shape}")
display(df.head())

Dataset loaded successfully! Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigree,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


Task 2: Data Preprocessing

In [4]:
# Handling Missing Values (In this dataset, 0s in biological metrics indicate missing data)
implausible_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[implausible_cols] = df[implausible_cols].replace(0, np.nan)

# Fill NaNs with the median value of each column
df.fillna(df.median(), inplace=True)
print("Missing values changed with median.")

Missing values changed with median.


In [5]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# 3. Train-Test Split (80% Train for GridSearch, 20% Holdout Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Data split into training ({X_train.shape[0]} samples) and testing ({X_test.shape[0]} samples).")

# 4. Standardizing the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features standardized using StandardScaler.")

Data split into training (614 samples) and testing (154 samples).
Features standardized using StandardScaler.


Task3: Deep Learning Grid Search

In [6]:
# defining a func to call with diff param combinations
def create_model(learning_rate=0.001, init_mode='glorot_uniform', activation='relu', dropout_rate=0.2, neurons=16):
    model = Sequential()

    # Input Layer & Hidden Layer 1 (Tuning: initialization, activation, neurons)
    model.add(Input(shape=(8,)))
    model.add(Dense(neurons, kernel_initializer=init_mode, activation=activation))

    # Regularization Layer (Tuning: dropout)
    model.add(Dropout(dropout_rate))

    # Output Layer (Binary Classification so we use Sigmoid)
    model.add(Dense(1, kernel_initializer=init_mode, activation='sigmoid'))

    # Compile Model (Tuning: learning rate)
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    return model

# Wrap the model in KerasClassifier
model = KerasClassifier(model=create_model, verbose=0)
print("Keras model wrapped successfully for Scikit-Learn.")


Keras model wrapped successfully for Scikit-Learn.


In [7]:
# all param combination (2 for each (64 total combination))
param_grid = {
    'batch_size': [16, 32],                    # Tune batch size
    'epochs': [25, 50],                       # Tune epochs
    'model__learning_rate': [0.001, 0.01],     # Tune learning rate
    'model__init_mode': ['uniform', 'glorot_uniform'], # Tune weight initialization
    'model__activation': ['relu', 'tanh'],     # Tune activation functions
    'model__dropout_rate': [0.1, 0.2],         # Tune dropout regularization
    'model__neurons': [16, 32]                 # Tune number of neurons
}

In [8]:
# Initialize GridSearchCV
# cv=2 means 2-fold cross-validation.
# Total fits = 128 combinations * 2 folds = 256 model training runs.
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=1, cv=2, verbose=2)

print("Starting Grid Search")
grid_result = grid.fit(X_train_scaled, y_train)

Starting Grid Search
Fitting 2 folds for each of 128 candidates, totalling 256 fits
[CV] END batch_size=16, epochs=25, model__activation=relu, model__dropout_rate=0.1, model__init_mode=uniform, model__learning_rate=0.001, model__neurons=16; total time=   4.3s
[CV] END batch_size=16, epochs=25, model__activation=relu, model__dropout_rate=0.1, model__init_mode=uniform, model__learning_rate=0.001, model__neurons=16; total time=   7.8s
[CV] END batch_size=16, epochs=25, model__activation=relu, model__dropout_rate=0.1, model__init_mode=uniform, model__learning_rate=0.001, model__neurons=32; total time=   7.8s
[CV] END batch_size=16, epochs=25, model__activation=relu, model__dropout_rate=0.1, model__init_mode=uniform, model__learning_rate=0.001, model__neurons=32; total time=   4.1s
[CV] END batch_size=16, epochs=25, model__activation=relu, model__dropout_rate=0.1, model__init_mode=uniform, model__learning_rate=0.01, model__neurons=16; total time=   2.5s
[CV] END batch_size=16, epochs=25, mo

/usr/local/lib/python3.13/dist-packages/numpy/ma/core.py:2881: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


In [9]:
# Print the best combination found
print(f"Best Accuracy: {grid_result.best_score_:.4f}")
print("Best Hyperparameters:")
for param, value in grid_result.best_params_.items():
    print(f" - {param.replace('model__', '')}: {value}")

Best Accuracy: 0.7883
Best Hyperparameters:
 - batch_size: 32
 - epochs: 50
 - activation: tanh
 - dropout_rate: 0.2
 - init_mode: glorot_uniform
 - learning_rate: 0.001
 - neurons: 16


In [10]:
# Use the best model found by GridSearch to predict the test set
best_model = grid_result.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print(f"Test Set Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))

Test Set Accuracy: 0.7078
Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.77      0.79      0.78       100
    Diabetes       0.59      0.56      0.57        54

    accuracy                           0.71       154
   macro avg       0.68      0.67      0.67       154
weighted avg       0.70      0.71      0.71       154

